In [ ]:
!pip install pyspark


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Medical_MapReduce_Colab") \
    .getOrCreate()

sc = spark.sparkContext


In [ ]:
medical_data = [
    ("P001", 72),
    ("P001", 75),
    ("P001", 70),
    ("P002", 80),
    ("P002", 78),
    ("P003", 90),
    ("P003", 92),
    ("P003", 88)
]

rdd = sc.parallelize(medical_data)
rdd.collect()


[('P001', 72),
 ('P001', 75),
 ('P001', 70),
 ('P002', 80),
 ('P002', 78),
 ('P003', 90),
 ('P003', 92),
 ('P003', 88)]

In [ ]:
rdd.map(lambda x: f"{x[0]},{x[1]}") \
   .saveAsTextFile("/content/medical_dataset")


In [ ]:
medical_rdd = sc.textFile("/content/medical_dataset")
medical_rdd.collect()


['P001,72',
 'P001,75',
 'P001,70',
 'P002,80',
 'P002,78',
 'P003,90',
 'P003,92',
 'P003,88']

In [ ]:
mapped_rdd = medical_rdd.map(
    lambda line: (line.split(",")[0], int(line.split(",")[1]))
)

mapped_rdd.collect()


[('P001', 72),
 ('P001', 75),
 ('P001', 70),
 ('P002', 80),
 ('P002', 78),
 ('P003', 90),
 ('P003', 92),
 ('P003', 88)]

In [ ]:
mapped_for_reduce = mapped_rdd.map(
    lambda x: (x[0], (x[1], 1))
)

mapped_for_reduce.collect()


[('P001', (72, 1)),
 ('P001', (75, 1)),
 ('P001', (70, 1)),
 ('P002', (80, 1)),
 ('P002', (78, 1)),
 ('P003', (90, 1)),
 ('P003', (92, 1)),
 ('P003', (88, 1))]

In [ ]:
reduced_rdd = mapped_for_reduce.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

reduced_rdd.collect()


[('P001', (217, 3)), ('P002', (158, 2)), ('P003', (270, 3))]

In [ ]:
average_hr_rdd = reduced_rdd.map(
    lambda x: (x[0], x[1][0] / x[1][1])
)

average_hr_rdd.collect()


[('P001', 72.33333333333333), ('P002', 79.0), ('P003', 90.0)]

In [ ]:
average_hr_rdd \
    .map(lambda x: f"{x[0]},{x[1]}") \
    .saveAsTextFile("/content/medical_results")


In [ ]:
medical_data = [
    ("P001", 72),
    ("P001", 75),
    ("P001", 70),
    ("P002", 80),
    ("P002", 78),
    ("P003", 90),
    ("P003", 92),
    ("P003", 88)
]

rdd = sc.parallelize(medical_data)

print("STEP 1: INPUT DATA")
print(rdd.collect())


STEP 1: INPUT DATA
[('P001', 72), ('P001', 75), ('P001', 70), ('P002', 80), ('P002', 78), ('P003', 90), ('P003', 92), ('P003', 88)]


Mapping key value pair

In [ ]:
mapped_rdd = rdd.map(lambda x: (x[0], x[1]))

print("\nSTEP 2: MAP OUTPUT (patient_id, heart_rate)")
print(mapped_rdd.collect())



STEP 2: MAP OUTPUT (patient_id, heart_rate)
[('P001', 72), ('P001', 75), ('P001', 70), ('P002', 80), ('P002', 78), ('P003', 90), ('P003', 92), ('P003', 88)]


MAP( Prepare for reduction)

In [ ]:
mapped_for_reduce = mapped_rdd.map(lambda x: (x[0], (x[1], 1)))

print("\nSTEP 3: MAP OUTPUT (prepare sum & count)")
print(mapped_for_reduce.collect())



STEP 3: MAP OUTPUT (prepare sum & count)
[('P001', (72, 1)), ('P001', (75, 1)), ('P001', (70, 1)), ('P002', (80, 1)), ('P002', (78, 1)), ('P003', (90, 1)), ('P003', (92, 1)), ('P003', (88, 1))]


# REDUCE PHASE – Aggregate Values

In [ ]:
reduced_rdd = mapped_for_reduce.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

print("\nSTEP 4: REDUCE OUTPUT (sum, count per patient)")
print(reduced_rdd.collect())



STEP 4: REDUCE OUTPUT (sum, count per patient)
[('P001', (217, 3)), ('P002', (158, 2)), ('P003', (270, 3))]


# FINAL MAP – Compute Average Heart Rate

In [ ]:
average_hr_rdd = reduced_rdd.map(
    lambda x: (x[0], round(x[1][0] / x[1][1], 2))
)

print("\nSTEP 5: FINAL MAP OUTPUT (average heart rate)")
print(average_hr_rdd.collect())



STEP 5: FINAL MAP OUTPUT (average heart rate)
[('P001', 72.33), ('P002', 79.0), ('P003', 90.0)]


# Appointment dataset

In [ ]:
# Columns: appointment_id, patient_id, doctor_id, appointment_date, status
appointment_data = [
    (1, 'P001', 'D001', '2026-02-01', 'Completed'),
    (2, 'P002', 'D001', '2026-02-01', 'Completed'),
    (3, 'P003', 'D002', '2026-02-02', 'Scheduled'),
    (4, 'P001', 'D003', '2026-02-03', 'Cancelled'),
    (5, 'P004', 'D002', '2026-02-03', 'Completed'),
    (6, 'P005', 'D001', '2026-02-04', 'Scheduled'),
    (7, 'P006', 'D003', '2026-02-04', 'Completed'),
]

rdd = sc.parallelize(appointment_data)

print("STEP 1: INPUT DATA (Appointments Table)")
print(rdd.collect())


STEP 1: INPUT DATA (Appointments Table)
[(1, 'P001', 'D001', '2026-02-01', 'Completed'), (2, 'P002', 'D001', '2026-02-01', 'Completed'), (3, 'P003', 'D002', '2026-02-02', 'Scheduled'), (4, 'P001', 'D003', '2026-02-03', 'Cancelled'), (5, 'P004', 'D002', '2026-02-03', 'Completed'), (6, 'P005', 'D001', '2026-02-04', 'Scheduled'), (7, 'P006', 'D003', '2026-02-04', 'Completed')]


In [ ]:
mapped_rdd = rdd.map(lambda x: (x[2], 1))  # x[2] = doctor_id

print("\nSTEP 2: MAP OUTPUT (doctor_id, 1 for each appointment)")
print(mapped_rdd.collect())



STEP 2: MAP OUTPUT (doctor_id, 1 for each appointment)
[('D001', 1), ('D001', 1), ('D002', 1), ('D003', 1), ('D002', 1), ('D001', 1), ('D003', 1)]


In [ ]:
mapped_for_reduce = mapped_rdd.map(lambda x: (x[0], x[1]))

print("\nSTEP 3: MAP PREPARATION (ready for reduce)")
print(mapped_for_reduce.collect())



STEP 3: MAP PREPARATION (ready for reduce)
[('D001', 1), ('D001', 1), ('D002', 1), ('D003', 1), ('D002', 1), ('D001', 1), ('D003', 1)]


In [ ]:
shuffled_rdd = mapped_for_reduce.groupByKey()

print("\nSTEP 4: SHUFFLE OUTPUT (appointments grouped by doctor)")
print([(k, list(v)) for k, v in shuffled_rdd.collect()])



STEP 4: SHUFFLE OUTPUT (appointments grouped by doctor)
[('D002', [1, 1]), ('D001', [1, 1, 1]), ('D003', [1, 1])]


In [ ]:
reduced_rdd = mapped_for_reduce.reduceByKey(lambda a, b: a + b)

print("\nSTEP 5: REDUCE OUTPUT (total appointments per doctor)")
print(reduced_rdd.collect())



STEP 5: REDUCE OUTPUT (total appointments per doctor)
[('D002', 2), ('D001', 3), ('D003', 2)]


In [ ]:
final_rdd = reduced_rdd.map(lambda x: f"Doctor {x[0]} has {x[1]} appointments")

print("\nSTEP 6: FINAL MAP OUTPUT (formatted results)")
print(final_rdd.collect())



STEP 6: FINAL MAP OUTPUT (formatted results)
['Doctor D002 has 2 appointments', 'Doctor D001 has 3 appointments', 'Doctor D003 has 2 appointments']
